In [1]:
!pip install --upgrade transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/10.5 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 22.4 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/4.2 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 82.7 MB/s eta 0:00:00


  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.4.2
    Uninstalling hf-xet-1.4.2:


...

    Found existing installation: transformers 4.57.1


    Uninstalling transformers-4.57.1:


      Successfully uninstalled transformers-4.57.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-tunix 0.1.7 requires transformers<=4.57.1, but you have transformers 5.7.0 which is incompatible.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# ── Cell 0: Setup ─────────────────────────────────────────────────────────────
# Expands neutral activations from 10 → 100 stories across diverse topics.
# Outputs: /kaggle/working/neutral_activations_expanded.npz
#   shape: [100, n_layers, d_model]  (first 10 rows = original neutrals)
#
# Usage in PCA notebook: load this file and use instead of resid_acts['__neutral__']

import os, gc, pickle
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ── Device: TPU → CUDA → CPU ───────────────────────────────────────────────────
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    device_type = "tpu"
    print(f"TPU available: {device}")
except (ImportError, RuntimeError):
    xm = None
    if torch.cuda.is_available():
        device = torch.device("cuda")
        device_type = "cuda"
    else:
        device = torch.device("cpu")
        device_type = "cpu"
    print(f"Using device: {device}")

# ── Paths ──────────────────────────────────────────────────────────────────────
MODEL_DIR = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1"
if not os.path.exists(MODEL_DIR):
    for candidate in [
        "/kaggle/input/gemma-4-e2b-it",
        "/kaggle/input/gemma4-e2b/transformers/gemma-4-e2b-it/1",
    ]:
        if os.path.exists(candidate):
            MODEL_DIR = candidate
            break

ACTS_PATH = (
    "/kaggle/input/notebooks/bencarson/gemma-4-functional-emotions-vector-extraction"
    "/emotions_phase1/activations.pkl"
)

NEUTRAL_PROMPT = (
    "Write a short factual description (3-4 sentences) about {topic}."
    " Be precise and informative.\n\nDescription:"
)
MAX_NEW_TOKENS = 100
OUTPUT_PATH    = "/kaggle/working/neutral_activations_expanded.npz"

print(f"Model dir: {MODEL_DIR}")


/usr/local/lib/python3.12/site-packages/torch_xla/__init__.py:258: UserWarning: `tensorflow` can conflict with `torch-xla`. Prefer `tensorflow-cpu` when using PyTorch/XLA. To silence this warning, `pip uninstall -y tensorflow && pip install tensorflow-cpu`. If you are in a notebook environment such as Colab or Kaggle, restart your notebook runtime afterwards.
  warnings.warn(


/tmp/ipykernel_73/1067821362.py:16: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


TPU available: xla:0
Model dir: /kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1


E0000 00:00:1777596762.740611      73 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


In [3]:
# ── Cell 1: New neutral topics (90) ───────────────────────────────────────────
# Diverse across domains to maximise neutral subspace coverage.
# Original 10 (water cycle, tectonic plates, etc.) are kept from the pkl.

NEW_NEUTRAL_TOPICS = [
    # Cooking / Food (10)
    "the Maillard reaction in cooking",
    "how pasta is made from wheat flour",
    "the process of cheese aging and ripening",
    "how bread rises through yeast fermentation",
    "the difference between simmering and boiling",
    "how olive oil is pressed from olives",
    "how salt preserves food from spoilage",
    "the structure and composition of a chicken egg",
    "how coffee beans are decaffeinated",
    "how chocolate is produced from cacao beans",
    # Geography / Earth (10)
    "how river deltas form at coastlines",
    "the formation of desert sand dunes",
    "how glaciers carve valleys and reshape landscapes",
    "how the water table works underground",
    "the formation of coral reefs in tropical oceans",
    "how volcanoes form and create new islands",
    "the jet stream and its effects on weather patterns",
    "how limestone caves form through erosion",
    "the difference between weather and climate",
    "how sedimentary rock layers form over time",
    # History / Technology (10)
    "how the printing press changed information distribution",
    "the construction of Roman aqueducts for water supply",
    "how medieval castles were designed and built",
    "the development of coinage in ancient economies",
    "how the compass changed maritime navigation",
    "the history of paper making in ancient China",
    "how ancient Egyptians preserved mummies",
    "the development of postal systems historically",
    "how the steam engine drove the industrial revolution",
    "the construction of railways in the nineteenth century",
    # Engineering / Technology (10)
    "how transistors amplify electrical signals",
    "the principle behind hydraulic braking systems",
    "how fiber optic cables transmit data using light",
    "how internal combustion engines work",
    "how radio waves are transmitted and received",
    "how mechanical locks and keys work",
    "the structural principles of suspension bridges",
    "how batteries convert chemical energy to electricity",
    "how a microwave oven heats food using radiation",
    "how solar panels convert sunlight to electricity",
    # Biology / Medicine (10)
    "how the immune system identifies and destroys pathogens",
    "the process of cell division in mitosis",
    "how vaccines train the immune system",
    "the structure and function of the human kidney",
    "how bones repair themselves after a fracture",
    "how insulin regulates blood sugar levels",
    "the process of digestion in the stomach and intestines",
    "how the human eye focuses light onto the retina",
    "how antibiotics work against bacterial infections",
    "the structure and function of a cell membrane",
    # Music / Arts (10)
    "how sound waves create musical pitch and tone",
    "the difference between major and minor musical scales",
    "how pigments mix in oil painting",
    "the structure and sections of a symphony orchestra",
    "how string instruments produce sound through resonance",
    "the history of the piano keyboard layout",
    "how architectural acoustics are designed for concert halls",
    "the principles of linear perspective in drawing",
    "how printing techniques evolved in the history of art",
    "how photography captures and reproduces images",
    # Sports / Games (5)
    "how aerodynamics affects a baseball pitch",
    "the rules and piece movements in chess",
    "the physics of a tennis ball bounce on different surfaces",
    "how swimming strokes differ in efficiency",
    "how Olympic swimming pool lanes are standardised",
    # Economics / Finance (10)
    "how compound interest accumulates over time",
    "the role of central banks in managing economies",
    "how supply and demand interact to set prices",
    "how stock exchanges facilitate share trading",
    "how inflation affects purchasing power over time",
    "the difference between GDP and GNP as economic measures",
    "how insurance pools risk across large populations",
    "how currency exchange rates are determined",
    "the concept of opportunity cost in economics",
    "how government bonds work as a form of debt",
    # Languages / Linguistics (5)
    "how related languages share a common ancestor",
    "the difference between syntax and semantics in grammar",
    "how alphabets developed from ancient pictographs",
    "how languages change and evolve across generations",
    "how the International Phonetic Alphabet works",
    # Architecture / Materials (5)
    "how concrete gains structural strength after mixing",
    "the principles of arch bridges and load distribution",
    "how glass is manufactured from silica sand",
    "how skyscrapers resist wind loads through design",
    "how building insulation reduces heat transfer",
    # Mathematics / Logic (5)
    "how prime numbers are distributed among integers",
    "the principles of Euclidean geometry and parallel lines",
    "how binary number systems represent data in computers",
    "the concept of probability in predicting outcomes",
    "how encryption algorithms protect digital information",
]

assert len(NEW_NEUTRAL_TOPICS) == 90, f"Expected 90 topics, got {len(NEW_NEUTRAL_TOPICS)}"
print(f"New neutral topics: {len(NEW_NEUTRAL_TOPICS)}")


New neutral topics: 90


In [4]:
# ── Cell 2: Load existing neutral activations from pkl ─────────────────────────
with open(ACTS_PATH, 'rb') as f:
    saved = pickle.load(f)

resid_acts = saved['resid']
neutral_existing = resid_acts['__neutral__']   # [10, n_layers, d_model]
n_layers = neutral_existing.shape[1]
d_model  = neutral_existing.shape[2]

print(f"Existing neutral activations: {neutral_existing.shape}")
print(f"n_layers={n_layers}, d_model={d_model}")

del saved, resid_acts
gc.collect()


Existing neutral activations: (10, 35, 1536)
n_layers=35, d_model=1536


60

In [5]:
# ── Cell 3: Load Gemma 4 E2B ───────────────────────────────────────────────────
print(f"Loading tokenizer from {MODEL_DIR} ...")
# extra_special_tokens={} works around a transformers version mismatch where
# the Gemma tokenizer config stores extra_special_tokens as a list but the
# installed transformers expects a dict and calls .keys() on it.
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, extra_special_tokens={})

print(f"Loading model onto {device_type} ...")
# device_map="auto" is CUDA-only — not supported on TPU/XLA.
# Load to CPU first, then move to target device.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.bfloat16,
)
model = model.to(device)
model.eval()
print("Model loaded.")

if xm is not None:
    xm.mark_step()   # flush initial XLA compilation graph


Loading tokenizer from /kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1 ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading model onto tpu ...


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Model loaded.


/tmp/ipykernel_73/3240394592.py:20: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()   # flush initial XLA compilation graph


In [6]:
# ── Cell 4: Build prompt texts (no generation needed) ─────────────────────────
# We extract activations from the prompt text only — no completion generated.
# The prompt is already factual neutral content ("Write a short factual
# description about X..."), which is sufficient to establish the neutral
# activation subspace. Avoids the DynamicSlidingWindowLayer XLA bug that
# occurs when sequence length < sliding_window size (512).

new_texts = [NEUTRAL_PROMPT.format(topic=topic) for topic in NEW_NEUTRAL_TOPICS]

print(f"Built {len(new_texts)} prompt texts (no generation).")
print(f"Example: {new_texts[0]!r}")


Built 90 prompt texts (no generation).
Example: 'Write a short factual description (3-4 sentences) about the Maillard reaction in cooking. Be precise and informative.\n\nDescription:'


In [7]:
# ── Cell 5: Extract last-token activations ────────────────────────────────────
# hidden_states[0] = embedding input; hidden_states[i+1] = after block i
# Equivalent to cache[f"blocks.{i}.hook_resid_post"][0, -1, :] in TransformerLens

def extract_last_token_acts(text, model, tokenizer, n_layers):
    """Return [n_layers, d_model] float32 numpy array — last-token residual stream."""
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=2048
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, use_cache=False)
    if xm is not None:
        xm.mark_step()   # materialise XLA graph before reading tensors
    acts = np.stack([
        outputs.hidden_states[i + 1][0, -1, :].float().cpu().numpy()
        for i in range(n_layers)
    ])  # [n_layers, d_model]
    return acts

new_neutral_acts = []
for i, text in enumerate(new_texts):
    acts = extract_last_token_acts(text, model, tokenizer, n_layers)
    new_neutral_acts.append(acts)
    if i % 10 == 0:
        print(f"  Extracted {i+1}/{len(new_texts)} ...")
        gc.collect()

new_neutral_acts = np.stack(new_neutral_acts)   # [90, n_layers, d_model]
print(f"New neutral activations: {new_neutral_acts.shape}")


/tmp/ipykernel_73/1774282634.py:13: DeprecationWarning: Use torch_xla.sync instead
  xm.mark_step()   # materialise XLA graph before reading tensors


  Extracted 1/90 ...


  Extracted 11/90 ...


  Extracted 21/90 ...


  Extracted 31/90 ...


...

  Extracted 51/90 ...


  Extracted 61/90 ...


  Extracted 71/90 ...


  Extracted 81/90 ...


New neutral activations: (90, 35, 1536)


In [8]:
# ── Cell 6: Combine with existing 10 and save ─────────────────────────────────
all_neutral = np.concatenate(
    [neutral_existing, new_neutral_acts], axis=0
)  # [100, n_layers, d_model]
print(f"Combined neutral activations: {all_neutral.shape}")

np.savez_compressed(OUTPUT_PATH, activations=all_neutral)
print(f"Saved to {OUTPUT_PATH}")

# Sanity check: norms should be similar across all 100 stories
norms = np.linalg.norm(all_neutral[:, 24, :], axis=-1)   # layer 24 ≈ layer 25 (0-indexed)
print(f"Layer 24 norms  min={norms.min():.1f}  max={norms.max():.1f}  mean={norms.mean():.1f}")
print(f"Original 10 mean: {norms[:10].mean():.1f}  New 90 mean: {norms[10:].mean():.1f}")


Combined neutral activations: (100, 35, 1536)


Saved to /kaggle/working/neutral_activations_expanded.npz
Layer 24 norms  min=61.6  max=85.2  mean=64.8
Original 10 mean: 77.9  New 90 mean: 63.3


In [9]:
# ── Cell 7: Verify — neutral PCA structure ────────────────────────────────────
from sklearn.decomposition import PCA
from scipy import stats

LAYER = 25   # 1-indexed in the PCA notebook; 0-indexed here = layer 24
layer_idx = LAYER - 1   # 0-indexed

neutral_at_layer = all_neutral[:, layer_idx, :]   # [100, d_model]

pca_n = PCA()
pca_n.fit(neutral_at_layer)
cumvar = np.cumsum(pca_n.explained_variance_ratio_)

print(f"Neutral PCA at layer {LAYER} (n=100):")
for thresh in [0.30, 0.40, 0.50, 0.60]:
    k = int(np.searchsorted(cumvar, thresh) + 1)
    print(f"  {thresh*100:.0f}% variance → {k} components")

print(f"\nTop-10 PC variance:")
for i in range(10):
    print(f"  PC{i+1}: {pca_n.explained_variance_ratio_[i]*100:.1f}%")


Neutral PCA at layer 25 (n=100):
  30% variance → 1 components
  40% variance → 1 components
  50% variance → 2 components
  60% variance → 3 components

Top-10 PC variance:
  PC1: 49.5%
  PC2: 6.3%
  PC3: 5.7%
  PC4: 4.7%
  PC5: 3.9%
  PC6: 3.6%
  PC7: 3.4%
  PC8: 2.7%
  PC9: 2.5%
  PC10: 2.3%
